# Capa Bronze
El objetivo de este notebook es leer los datos crudos (Raw), aplicar limpieza puramente tecnica (como nombres de columnas compatibles), agregar metadatos de linaje (fecha de ingesta y archivo de origen) y guardar el resultado como una tabla Delta.

In [ ]:
from pyspark.sql.functions import current_timestamp, input_file_name

# 1. Definir la ruta donde se depositaron los archivos Raw
raw_place = "/Volumes/workspace/supermarket/supermarket_data/supermarket_data_kaggle"
raw_file_path = f"{raw_place}/supermarket.csv"


In [ ]:
# 2. Leer los datos crudos desde el volumen
df_bronze = (spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(raw_file_path))

In [ ]:
# 3. Limpieza tecnica: Reemplazar espacios por guiones bajos en los nombres de columnas
for col_name in df_bronze.columns:
    df_bronze = df_bronze.withColumnRenamed(col_name, col_name.replace(" ", "_"))

In [ ]:
# 4. Agregar Metadatos (Linaje de los datos)
df_bronze = (df_bronze
    .withColumn("ingestion_timestamp", current_timestamp())  # Cuando se proceso
    .withColumn("source_file", input_file_name())            # De que archivo viene
)

In [ ]:
# 5. Guardar como tabla Bronze definitiva (en formato Delta) en el volumen
bronze_volume_path = "/Volumes/workspace/supermarket/supermarket_data/bronze/supermarket_bronze"
(df_bronze.write
    .format("delta")
    .mode("overwrite")
    .save(bronze_volume_path))

print(f"Tabla supermarket_bronze guardada exitosamente en el volumen: {bronze_volume_path}")

In [ ]:
# 6. Verificacion rapida de los datos guardados en el volumen
display(spark.read.format("delta").load(bronze_volume_path).limit(5))